<a href="https://colab.research.google.com/github/tangitapkullaniyor/CENG467_Midterm_290201060/blob/main/Question3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers evaluate rouge_score nltk bert-score networkx scikit-learn -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset_sum = load_dataset("cnn_dailymail", "3.0.0")

print(dataset_sum)
print(dataset_sum["test"][0].keys())
print(dataset_sum["test"][0]["article"][:500])
print(dataset_sum["test"][0]["highlights"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})
dict_keys(['article', 'highlights', 'id'])
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, includin
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and t

In [3]:
test_data = dataset_sum["test"].select(range(50))

In [7]:
import nltk
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
import numpy as np
import networkx as nx
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def textrank_summary(text, num_sentences=3):
    sentences = sent_tokenize(text)

    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer(stop_words="english")
    sentence_vectors = vectorizer.fit_transform(sentences)

    similarity_matrix = cosine_similarity(sentence_vectors)

    graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(graph)

    ranked_sentences = sorted(
        ((scores[i], sentence) for i, sentence in enumerate(sentences)),
        reverse=True
    )

    selected_sentences = [sentence for _, sentence in ranked_sentences[:num_sentences]]

    return " ".join(selected_sentences)

In [9]:
!pip install sentencepiece -q

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_checkpoint = "t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_checkpoint)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_checkpoint)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
def t5_summary(text, max_input_length=512, max_output_length=80):
    input_text = "summarize: " + text

    inputs = t5_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    summary_ids = t5_model.generate(
        inputs["input_ids"],
        max_length=max_output_length,
        min_length=30,
        num_beams=4,
        early_stopping=True
    )

    return t5_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [13]:
for i in range(3):
    article = test_data[i]["article"]
    reference = test_data[i]["highlights"]

    textrank_sum = textrank_summary(article, num_sentences=3)
    t5_sum = t5_summary(article)

    print("=" * 80)
    print(f"EXAMPLE {i+1}")
    print("=" * 80)

    print("\nARTICLE SNIPPET:")
    print(article[:500])

    print("\nREFERENCE SUMMARY:")
    print(reference)

    print("\nTEXTRANK SUMMARY:")
    print(textrank_sum)

    print("\nT5 SUMMARY:")
    print(t5_sum)

    print("\n")

EXAMPLE 1

ARTICLE SNIPPET:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, includin

REFERENCE SUMMARY:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

TEXTRANK SUMMARY:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territ

In [14]:
!pip install evaluate rouge_score bert-score nltk -q

In [15]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [16]:
references = []
textrank_preds = []
t5_preds = []

for example in test_data:
    article = example["article"]
    reference = example["highlights"]

    textrank_sum = textrank_summary(article)
    t5_sum = t5_summary(article)

    references.append(reference)
    textrank_preds.append(textrank_sum)
    t5_preds.append(t5_sum)

In [17]:
rouge_textrank = rouge.compute(predictions=textrank_preds, references=references)
rouge_t5 = rouge.compute(predictions=t5_preds, references=references)

print("TextRank ROUGE:", rouge_textrank)
print("T5 ROUGE:", rouge_t5)

TextRank ROUGE: {'rouge1': np.float64(0.26671161864876747), 'rouge2': np.float64(0.08622481596991599), 'rougeL': np.float64(0.1829013547940424), 'rougeLsum': np.float64(0.22146231124772248)}
T5 ROUGE: {'rouge1': np.float64(0.31523291569722567), 'rouge2': np.float64(0.12964203520120254), 'rougeL': np.float64(0.23860301336733508), 'rougeLsum': np.float64(0.2683042424008064)}


In [18]:
bleu_textrank = bleu.compute(
    predictions=textrank_preds,
    references=[[ref] for ref in references]
)

bleu_t5 = bleu.compute(
    predictions=t5_preds,
    references=[[ref] for ref in references]
)

print("TextRank BLEU:", bleu_textrank)
print("T5 BLEU:", bleu_t5)

TextRank BLEU: {'bleu': 0.055911839269342165, 'precisions': [0.212, 0.061772151898734175, 0.03461538461538462, 0.021558441558441558], 'brevity_penalty': 1.0, 'length_ratio': 2.1130480718436346, 'translation_length': 4000, 'reference_length': 1893}
T5 BLEU: {'bleu': 0.09521985183851381, 'precisions': [0.2994768962510898, 0.10472370766488413, 0.06244302643573382, 0.04197761194029851], 'brevity_penalty': 1.0, 'length_ratio': 1.2118330692023243, 'translation_length': 2294, 'reference_length': 1893}


In [19]:
meteor_textrank = meteor.compute(
    predictions=textrank_preds,
    references=references
)

meteor_t5 = meteor.compute(
    predictions=t5_preds,
    references=references
)

print("TextRank METEOR:", meteor_textrank)
print("T5 METEOR:", meteor_t5)

TextRank METEOR: {'meteor': np.float64(0.2966459603493041)}
T5 METEOR: {'meteor': np.float64(0.3063932574591908)}


In [20]:
bertscore_textrank = bertscore.compute(
    predictions=textrank_preds,
    references=references,
    lang="en"
)

bertscore_t5 = bertscore.compute(
    predictions=t5_preds,
    references=references,
    lang="en"
)

print("TextRank BERTScore F1:", sum(bertscore_textrank["f1"]) / len(bertscore_textrank["f1"]))
print("T5 BERTScore F1:", sum(bertscore_t5["f1"]) / len(bertscore_t5["f1"]))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TextRank BERTScore F1: 0.8597430980205536
T5 BERTScore F1: 0.8673081910610199
